# **Logging & Debugging**

---

## Why Not `print()`

`print` has no levels, no timestamps, no module names, and no way to silence it in production. Python's `logging` module gives you all of that plus per-module control. The rule of thumb:

* **Library/training code** — get a logger with `logging.getLogger(__name__)` and never call `basicConfig`.
* **The application entry point** (your `main`/script) — configure handlers, levels, and formatting **once**.

This way the same library code is quiet when imported and verbose when you turn it up.

## Configuring `logging`

```python
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

log.info("epoch %d  loss=%.4f  lr=%.2e", epoch, loss, lr)
log.warning("grad norm %.1f exceeds clip threshold", grad_norm)
```

**Structured logs** (one JSON object per line) are easier for log aggregators (Loki, ELK, CloudWatch) to parse:

```python
import json, logging

class JsonFormatter(logging.Formatter):
    def format(self, record):
        return json.dumps({
            "ts": self.formatTime(record),
            "level": record.levelname,
            "logger": record.name,
            "msg": record.getMessage(),
        })

h = logging.StreamHandler()
h.setFormatter(JsonFormatter())
logging.getLogger().addHandler(h)
```

## Catching NaNs and Bad Gradients

The most common training failures are silent: a loss becomes `NaN`, or a gradient explodes, and you only notice epochs later. PyTorch's **anomaly detection** turns the *first* bad op into an immediate, traceable error and points back to the forward op that produced it.

```python
import torch

# Global switch (slow — use only while debugging):
torch.autograd.set_detect_anomaly(True)

# Or scope it to one backward pass:
with torch.autograd.detect_anomaly():
    loss = model(x).sum()
    loss.backward()   # raises at the op that emitted NaN/Inf, with forward traceback
```

Manual guards are cheap and worth keeping in:

```python
if not torch.isfinite(loss):
    log.error("non-finite loss=%s at step %d", loss.item(), step)
    raise FloatingPointError("loss is NaN/Inf")

torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
```

## Common Bugs & How They Show Up

| Symptom | Likely cause |
|---------|--------------|
| Loss is `NaN` after a few steps | learning rate too high; `log(0)`/`sqrt(neg)`; missing `eps`; exploding grads |
| Loss flat / not decreasing | wrong reduction, detached graph (`.item()`/`.detach()` in the loss path), frozen params |
| `RuntimeError: shape mismatch` | a forgotten transpose, wrong `dim` in `cat`/`view`, batch vs. feature axis swapped |
| Silent broadcasting | `(N,1)` vs `(N,)` in a loss — prints fine but trains wrong |
| CUDA error “device-side assert” | label index out of `[0, num_classes)` range in `CrossEntropyLoss` |

Printing shapes and dtypes at the boundary catches most of these:

```python
log.debug("x %s %s | logits %s", tuple(x.shape), x.dtype, tuple(logits.shape))
```

## Interactive Debugging

Drop into a debugger at any line with the builtin `breakpoint()` (Python 3.7+), which respects the `PYTHONBREAKPOINT` env var (set it to `0` to disable all breakpoints in production):

```python
def training_step(batch):
    loss = compute_loss(batch)
    breakpoint()          # opens pdb here; inspect loss, batch, tensors
    return loss
```

Useful `pdb` commands: `n` (next), `s` (step into), `c` (continue), `p expr` / `pp expr` (print), `w` (where / stack), `u`/`d` (move up/down the stack), `q` (quit).

For post-mortem debugging of an uncaught exception, run `python -m pdb -c continue script.py`.

## References

* `logging` HOWTO: https://docs.python.org/3/howto/logging.html
* `torch.autograd.set_detect_anomaly`: https://pytorch.org/docs/stable/autograd.html#torch.autograd.set_detect_anomaly
* `pdb`: https://docs.python.org/3/library/pdb.html
* `breakpoint()` / `PYTHONBREAKPOINT`: https://docs.python.org/3/library/functions.html#breakpoint